## Output Structure

In [5]:
from pydantic import BaseModel
from typing import List

class DayPlan (BaseModel):
    day: str
    topic: str
    duration_minutes: int

class StudyPlan(BaseModel):
    goal: str
    days: list[DayPlan]

## Simple Agent (only LLM)

In [ ]:
from openai import OpenAI
import openai
openai.api_key = 
client = OpenAI(api_key=openai.api_key)

def study_planner(goal: str, max_min_per_day: int) -> StudyPlan:
    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": """
You are a study planning assistant.
Create a simple 5-day study plan.
Each day should have a topic and duration.
Each day must not exceed {max_min_per_day} minutes.
"""
            },
            {
                "role": "user",
                "content": goal
            }
        ],
        text_format=StudyPlan
        )

    return response.output_parsed

In [34]:
plan = study_planner("I want to learn AI agents in 5 days, 1 hour per day",  max_min_per_day=30)

print(plan.model_dump_json(indent=2))

{
  "goal": "Learn AI agents",
  "days": [
    {
      "day": "Day 1",
      "topic": "Introduction to AI Agents and Basic Concepts",
      "duration_minutes": 60
    },
    {
      "day": "Day 2",
      "topic": "Types of AI Agents and Environment Models",
      "duration_minutes": 60
    },
    {
      "day": "Day 3",
      "topic": "Agent Architectures and Rationality",
      "duration_minutes": 60
    },
    {
      "day": "Day 4",
      "topic": "Learning Agents and Adaptive Behavior",
      "duration_minutes": 60
    },
    {
      "day": "Day 5",
      "topic": "Applications and Case Studies of AI Agents",
      "duration_minutes": 60
    }
  ]
}


## Add Tools:
1. validate study time

In [24]:
def validate_study_time(days, max_min_per_day):
    problems = []
    for day in days:
        if day.duration_minutes > max_min_per_day:
            problems.append( f"{day.day}: {day.duration_minutes} minutes exceeds limit of {max_min_per_day}" )
    return {
        "is_valid" : len(problems) == 0,
        "problems" : problems
    }


In [25]:
validation = validate_study_time(plan.days, max_min_per_day=40)
print(validation)
print(plan.model_dump_json(indent=2))


{'is_valid': False, 'problems': ['Day 1: 60 minutes exceeds limit of 40', 'Day 2: 60 minutes exceeds limit of 40', 'Day 3: 60 minutes exceeds limit of 40', 'Day 4: 60 minutes exceeds limit of 40', 'Day 5: 60 minutes exceeds limit of 40']}
{
  "goal": "Learn AI agents in 5 days",
  "days": [
    {
      "day": "Day 1",
      "topic": "Introduction to AI Agents and Basic Concepts",
      "duration_minutes": 60
    },
    {
      "day": "Day 2",
      "topic": "Types of AI Agents and Environment Interaction",
      "duration_minutes": 60
    },
    {
      "day": "Day 3",
      "topic": "Agent Architectures and Design Approaches",
      "duration_minutes": 60
    },
    {
      "day": "Day 4",
      "topic": "Implementation of Simple AI Agents",
      "duration_minutes": 60
    },
    {
      "day": "Day 5",
      "topic": "Advanced Topics and Applications of AI Agents",
      "duration_minutes": 60
    }
  ]
}


## Revise Study Plan

In [ ]:
def revise_study_plan(goal: str, old_plan: StudyPlan, problems:list, max_min_per_day:int) -> StudyPlan:
    response = client.responses.parse( 
        model = "gpt-4o-mini",
        input = [
            {
                "role" : "system",
                "content" : """ You revise study plans.
Fix all validation problems.
No day may exceed {max_min_per_day} minutes.
Return a corrected plan."""
            },
            {
                "role": "user",
                "content": f"""
Goal:
{goal}

Old plan:
{old_plan.model_dump_json(indent=2)}

Problems:
{problems}
"""
            }
        ],
        text_format=StudyPlan       
    )
    return response.output_parsed
goal="I want to learn AI agents in 5 days."
revised_plan = revise_study_plan(goal, plan, validation["problems"], max_min_per_day=40)
print(revised_plan.model_dump_json(indent=2))



{
  "goal": "Learn AI agents in 5 days",
  "days": [
    {
      "day": "Day 1",
      "topic": "Introduction to AI Agents and Basic Concepts",
      "duration_minutes": 40
    },
    {
      "day": "Day 2",
      "topic": "Types of AI Agents and Environment Interaction",
      "duration_minutes": 40
    },
    {
      "day": "Day 3",
      "topic": "Agent Architectures and Design Approaches",
      "duration_minutes": 40
    },
    {
      "day": "Day 4",
      "topic": "Implementation of Simple AI Agents",
      "duration_minutes": 40
    },
    {
      "day": "Day 5",
      "topic": "Advanced Topics and Applications of AI Agents",
      "duration_minutes": 40
    }
  ]
}


In [40]:
def study_planner_agent(goal: str, max_min_per_day: int, max_iterations: int = 3):
    plan = study_planner(goal, max_min_per_day)

    for i in range(max_iterations):
        validation = validate_study_time(plan.days, max_min_per_day)

        if validation["is_valid"]:
            return {
                "plan": plan,
                "validation": validation,
                "iterations": i + 1
            }

        plan = revise_study_plan(
            goal=goal,
            old_plan=plan,
            problems=validation["problems"],
            max_min_per_day=max_min_per_day
        )

    return {
        "plan": plan,
        "validation": validate_study_time(plan.days, max_min_per_day),
        "iterations": max_iterations
    }

In [41]:
result = study_planner_agent(
    goal="I want to learn AI agents in 5 days.",
    max_min_per_day=30
)

print(result["plan"].model_dump_json(indent=2))
print(result["validation"])
print("Iterations:", result["iterations"])

{
  "goal": "Learn AI agents in 5 days",
  "days": [
    {
      "day": "Day 1",
      "topic": "Introduction to AI Agents and their Types",
      "duration_minutes": 30
    },
    {
      "day": "Day 2",
      "topic": "Agent Environments and Rationality",
      "duration_minutes": 30
    },
    {
      "day": "Day 3",
      "topic": "Search Algorithms for AI Agents",
      "duration_minutes": 30
    },
    {
      "day": "Day 4",
      "topic": "Learning and Adaptation in AI Agents",
      "duration_minutes": 30
    },
    {
      "day": "Day 5",
      "topic": "Implementing Simple AI Agents and Case Studies",
      "duration_minutes": 30
    }
  ]
}
{'is_valid': True, 'problems': []}
Iterations: 2


## Add Memory

In [42]:
import json
from pathlib import Path
MEMORY_FILE = Path("study_memory.json")

def load_memory():
    if MEMORY_FILE.exists():
        with open(MEMORY_FILE,"r") as f:
            return json.load(f)
    return {
        "completed_days": [],
        "notes": []
    }
def save_memory(memory):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)


def mark_day_completed(day: str, note: str = ""):
    memory = load_memory()

    if day not in memory["completed_days"]:
        memory["completed_days"].append(day)

    if note:
        memory["notes"].append({
            "day": day,
            "note": note
        })

    save_memory(memory)
    return memory


In [44]:
mark_day_completed("Day 1", "I understood what LLMs and tools are.")

{'completed_days': ['Day 1'],
 'notes': [{'day': 'Day 1', 'note': 'I understood what LLMs and tools are.'}]}

In [48]:
openai.__version__

'2.33.0'

In [45]:
print(load_memory())

{'completed_days': ['Day 1'], 'notes': [{'day': 'Day 1', 'note': 'I understood what LLMs and tools are.'}]}


In [ ]:
def create_study_plan_with_memory(goal: str, max_min_per_day: int) -> StudyPlan:
    memory = load_memory()

    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": f"""
You are a study planning agent.

Create a 5-day study plan.
Each day must not exceed {max_min_per_day} minutes.

Use the learner's memory to avoid repeating completed topics.
"""
            },
            {
                "role": "user",
                "content": f"""
Goal:
{goal}

Learner memory:
{json.dumps(memory, indent=2)}
"""
            }
        ],
        text_format=StudyPlan
    )

    return response.output_parsed

In [47]:
pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]
Note: you may need to restart the kernel to use updated packages.
